# Beyond Shortcuts: Multimodal Fake News Detection
## Notebook 01: Data Ingestion, Resilient Download, Deduplication & Group-Aware Splits

**Pipeline Goals (Phase A1):**
1. Resilient threaded image downloading with instant resume (skips already downloaded files).
2. Silence warning spam and prevent PIL decompression bomb hangs.
3. Deduplicate near-identical images via 64-bit difference hashing (`dhash`) and Hamming distance threshold (<= 4).
4. Prevent data leakage via group-aware splits (`group_id`).
5. Enforce strict data contract and export memory-optimized `table.parquet`.

### Cell 1: Imports, Warning Suppression & Configuration

In [8]:
import io
import logging
import os
import warnings
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path
from typing import Any, Dict, List, Optional, Set, Tuple

import numpy as np
import pandas as pd
from PIL import Image
import requests
from requests.adapters import HTTPAdapter
from tqdm import tqdm
import urllib3
from urllib3.util import Retry

# 1. Prevent PIL from pausing or warning on giant images
Image.MAX_IMAGE_PIXELS = None
warnings.simplefilter('ignore', Image.DecompressionBombWarning)
warnings.filterwarnings('ignore', category=UserWarning, module='PIL')

# 2. Silence urllib3 pool and dead-domain connection warnings
urllib3.disable_warnings()
logging.getLogger('urllib3').setLevel(logging.ERROR)
logging.getLogger('requests').setLevel(logging.ERROR)

# ==========================================
# Configuration & Hyperparameters
# ==========================================
INPUT_TSV_PATH: str = '/kaggle/input/datasets/jarroud/fake-news/multimodal_train.tsv'
OUTPUT_DIR: str = '/kaggle/working/images_v1'
OUTPUT_PARQUET_PATH: str = '/kaggle/working/table.parquet'

# 90,000 raw samples -> yields ~85,000 clean images (matches course spec: ~80k)
SAMPLE_SIZE: Optional[int] = 90000

TIMEOUT_SECONDS: int = 8
NUM_WORKERS: int = 32
DHASH_SIZE: int = 8
HAMMING_MAX_DISTANCE: int = 4
TRAIN_RATIO: float = 0.70
VAL_RATIO: float = 0.15
RANDOM_SEED: int = 42


SyntaxError: (unicode error) 'unicodeescape' codec can't decode bytes in position 2-3: truncated \UXXXXXXXX escape (3813600100.py, line 31)

### Cell 2: Dataset Loading & Filtering

In [ ]:
def load_and_filter_tsv(
    tsv_path: str,
    sample_size: Optional[int] = None,
) -> Tuple[pd.DataFrame, int]:
    """Load Fakeddit TSV file, drop rows with missing id/image_url, and optionally sample."""
    if not os.path.exists(tsv_path):
        alt_path = "/kaggle/input/fakeddit-multimodal-fake-news-classification/multimodal_only_samples/multimodal_train.tsv"
        if os.path.exists(alt_path):
            tsv_path = alt_path
        else:
            raise FileNotFoundError(f"Input TSV dataset not found at: {tsv_path}")

    print(f"Loading raw dataset from {tsv_path}...")
    df = pd.read_csv(tsv_path, sep="\t")
    total_raw_rows = len(df)

    df = df.dropna(subset=["id", "image_url"]).copy()
    df["id"] = df["id"].astype(str)

    if sample_size is not None and sample_size > 0:
        print(f"Applying SAMPLE_SIZE slice: taking first {sample_size:,} rows.")
        df = df.head(sample_size).copy()

    return df.reset_index(drop=True), total_raw_rows

### Cell 3: Resilient Threaded Downloader with Instant Salvage

In [ ]:
def fetch(
    url: str,
    out_path: str,
    session: requests.Session,
    timeout: int = 8,
) -> bool:
    """Download image, verify deeply, resize, and save as JPEG."""
    # 1. FAST RESUME: Skip if already downloaded and valid on disk
    if os.path.exists(out_path) and os.path.getsize(out_path) > 0:
        return True

    try:
        r = session.get(url, timeout=timeout)
        r.raise_for_status()

        # Guard: reject files > 25 MB
        if len(r.content) > 25_000_000:
            return False

        # Guard: check dimensions in header before decompressing pixels
        with Image.open(io.BytesIO(r.content)) as im_check:
            w, h = im_check.size
            if w * h > 50_000_000:
                return False  # Skip decompression bombs instantly

        im = Image.open(io.BytesIO(r.content)).convert("RGB")
        im.thumbnail((256, 256))
        im.save(out_path, "JPEG", quality=90)

        # Deep Image Verification: ensure complete, untruncated file
        with Image.open(out_path) as img_verify:
            img_verify.verify()
        with Image.open(out_path) as img_load:
            img_load.load()

        return True
    except Exception:
        if os.path.exists(out_path):
            try:
                os.remove(out_path)
            except OSError:
                pass
        return False


def download_all(rows: List[Dict[str, Any]], workers: int = 32) -> List[bool]:
    """Download with as_completed so progress bar never freezes on slow URLs."""
    session = requests.Session()
    retries = Retry(
        total=2,
        connect=1,   # Don't waste time retrying non-existent hostnames
        read=1,
        status=2,
        backoff_factor=0.3,
        status_forcelist=[429, 500, 502, 503, 504],
        raise_on_status=False,
    )
    adapter = HTTPAdapter(
        pool_connections=workers,
        pool_maxsize=workers,
        max_retries=retries,
    )
    session.mount("http://", adapter)
    session.mount("https://", adapter)
    session.headers.update(
        {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) MultimodalFakeNews/1.0"}
    )

    results = [False] * len(rows)
    with session:
        with ThreadPoolExecutor(max_workers=workers) as ex:
            future_to_idx = {
                ex.submit(fetch, r["image_url"], r["path"], session, TIMEOUT_SECONDS): idx
                for idx, r in enumerate(rows)
            }
            for future in tqdm(
                as_completed(future_to_idx),
                total=len(rows),
                desc="Downloading images",
            ):
                idx = future_to_idx[future]
                try:
                    results[idx] = future.result()
                except Exception:
                    results[idx] = False

    return results


def execute_image_downloads(
    df: pd.DataFrame,
    out_dir: str,
    workers: int = 32,
) -> pd.DataFrame:
    """Execute threaded downloads for valid rows and drop download failures."""
    os.makedirs(out_dir, exist_ok=True)
    df["image_path"] = df["id"].apply(lambda img_id: os.path.join(out_dir, f"{img_id}.jpg"))

    tasks = [{"image_url": u, "path": p} for u, p in zip(df["image_url"], df["image_path"])]
    print(f"Starting resilient download of {len(tasks):,} images with {workers} workers...")
    success_flags = download_all(tasks, workers=workers)

    df_success = df[success_flags].copy().reset_index(drop=True)
    return df_success


### Cell 4: O(log N) Perceptual Hashing & Fast Deduplication

In [ ]:
def dhash(path: str, size: int = 8) -> int:
    """64-bit difference hash: identical images share it, near-duplicates differ by few bits."""
    resample = getattr(Image, "Resampling", Image).BILINEAR
    im = Image.open(path).convert("L").resize((size + 1, size), resample)
    a = np.asarray(im, dtype=np.int16)
    bits = (a[:, 1:] > a[:, :-1]).flatten()
    return int("".join("1" if b else "0" for b in bits), 2)


def hamming(a: int, b: int) -> int:
    """Count differing bits between two 64-bit integers."""
    return bin(a ^ b).count("1")


def _hash_chunks(h: int) -> Tuple[int, ...]:
    """Split 64-bit hash into 5 disjoint chunks (pigeonhole guarantee for distance <= 4)."""
    return (
        h & 0x1FFF,          # bits 0-12 (13 bits)
        (h >> 13) & 0x1FFF,  # bits 13-25 (13 bits)
        (h >> 26) & 0x1FFF,  # bits 26-38 (13 bits)
        (h >> 39) & 0x1FFF,  # bits 39-51 (13 bits)
        (h >> 52) & 0xFFF,   # bits 52-63 (12 bits)
    )


class DisjointSet:
    """Disjoint Set Union (Union-Find) with path compression."""

    def __init__(self, size: int) -> None:
        self.parent = list(range(size))

    def find(self, i: int) -> int:
        if self.parent[i] != i:
            self.parent[i] = self.find(self.parent[i])
        return self.parent[i]

    def union(self, i: int, j: int) -> None:
        root_i, root_j = self.find(i), self.find(j)
        if root_i != root_j:
            self.parent[root_i] = root_j


def cluster_hashes(hashes: List[int], max_dist: int = 4) -> List[int]:
    """Cluster near-duplicate hashes in sublinear lookup time."""
    n = len(hashes)
    dsu = DisjointSet(n)
    buckets: List[Dict[int, List[int]]] = [{} for _ in range(5)]
    seen_pairs: Set[Tuple[int, int]] = set()

    for idx in tqdm(range(n), desc="Clustering duplicate groups"):
        h = hashes[idx]
        for c_idx, chunk in enumerate(_hash_chunks(h)):
            for match_idx in buckets[c_idx].setdefault(chunk, []):
                pair = (match_idx, idx)
                if pair not in seen_pairs:
                    seen_pairs.add(pair)
                    if hamming(h, hashes[match_idx]) <= max_dist:
                        dsu.union(match_idx, idx)
            buckets[c_idx][chunk].append(idx)

    root_map: Dict[int, int] = {}
    return [root_map.setdefault(dsu.find(i), len(root_map)) for i in range(n)]

### Cell 5: Group-Aware Splitting & Memory-Efficient Formatting

In [ ]:
def group_aware_split(
    df: pd.DataFrame,
    group_col: str = "group_id",
    train_ratio: float = 0.70,
    val_ratio: float = 0.15,
    seed: int = 42,
) -> pd.Series:
    """Split data into train, validate, and test respecting group boundaries."""
    rng = np.random.default_rng(seed)
    unique_groups = df[group_col].unique()
    rng.shuffle(unique_groups)

    group_sizes = df[group_col].value_counts().to_dict()
    total_samples = len(df)
    train_target = total_samples * train_ratio
    val_target = total_samples * val_ratio

    split_map: Dict[int, str] = {}
    curr_train = 0
    curr_val = 0

    for gid in unique_groups:
        size = group_sizes[gid]
        if curr_train < train_target:
            split_map[gid] = "train"
            curr_train += size
        elif curr_val < val_target:
            split_map[gid] = "validate"
            curr_val += size
        else:
            split_map[gid] = "test"

    return df[group_col].map(split_map)


def validate_splits(df: pd.DataFrame, group_col: str = "group_id", split_col: str = "split") -> None:
    """Assert zero group overlap between train, validate, and test partitions."""
    train_g = set(df.loc[df[split_col] == "train", group_col])
    val_g = set(df.loc[df[split_col] == "validate", group_col])
    test_g = set(df.loc[df[split_col] == "test", group_col])

    assert len(train_g & val_g) == 0, f"Leakage: {len(train_g & val_g)} groups shared train/val!"
    assert len(train_g & test_g) == 0, f"Leakage: {len(train_g & test_g)} groups shared train/test!"
    assert len(val_g & test_g) == 0, f"Leakage: {len(val_g & test_g)} groups shared val/test!"


def format_contract(df: pd.DataFrame) -> pd.DataFrame:
    """Rename columns according to contract, cast to memory-efficient types."""
    rename_cols = {
        "clean_title": "text",
        "2_way_label": "label2",
        "6_way_label": "label6",
    }
    df = df.rename(columns=rename_cols)

    if "text" not in df.columns and "title" in df.columns:
        df["text"] = df["title"]

    df["is_mined_mismatch"] = False

    cols = ["id", "text", "image_path", "label2", "label6", "subreddit", "group_id", "split", "is_mined_mismatch"]
    df_filtered = df[cols].copy()

    df_filtered["id"] = df_filtered["id"].astype("string")
    df_filtered["text"] = df_filtered["text"].astype("string")
    df_filtered["image_path"] = df_filtered["image_path"].astype("string")
    df_filtered["label2"] = df_filtered["label2"].astype(np.int8)
    df_filtered["label6"] = df_filtered["label6"].astype(np.int8)
    df_filtered["subreddit"] = df_filtered["subreddit"].astype("category")
    df_filtered["group_id"] = df_filtered["group_id"].astype(np.int32)
    df_filtered["split"] = df_filtered["split"].astype("category")
    df_filtered["is_mined_mismatch"] = df_filtered["is_mined_mismatch"].astype(bool)

    return df_filtered

### Cell 6: Run Pipeline & Export Parquet Contract

In [ ]:
def run_pipeline() -> None:
    # 1. Load and filter
    df, total_raw = load_and_filter_tsv(INPUT_TSV_PATH, sample_size=SAMPLE_SIZE)
    total_valid = len(df)

    # 2. Resilient download (instantly skips existing images on disk)
    df_downloaded = execute_image_downloads(df, OUTPUT_DIR, workers=NUM_WORKERS)

    # 3. Deduplication
    hashes = [dhash(p, size=DHASH_SIZE) for p in tqdm(df_downloaded["image_path"], desc="Computing dhashes")]
    df_downloaded["group_id"] = cluster_hashes(hashes, max_dist=HAMMING_MAX_DISTANCE)

    # 4. Group Split
    df_downloaded["split"] = group_aware_split(
        df_downloaded,
        group_col="group_id",
        train_ratio=TRAIN_RATIO,
        val_ratio=VAL_RATIO,
        seed=RANDOM_SEED,
    )
    validate_splits(df_downloaded, group_col="group_id", split_col="split")

    # 5. Format and Export Parquet
    final_df = format_contract(df_downloaded)
    final_df.to_parquet(OUTPUT_PARQUET_PATH, index=False, engine="pyarrow")

    # Summary
    downloaded = len(final_df)
    attrition = ((total_valid - downloaded) / total_valid * 100) if total_valid > 0 else 0
    print("\n" + "=" * 50)
    print("PHASE A1 COMPLETE")
    print("=" * 50)
    print(f"Total processed:   {total_valid:,}")
    print(f"Downloaded images: {downloaded:,}")
    print(f"Dead link losses:  {total_valid - downloaded:,} ({attrition:.2f}% attrition)")
    print(f"Unique groups:     {final_df['group_id'].nunique():,}")
    print("\nSplit Distribution:")
    print(final_df["split"].value_counts())
    print("=" * 50)


run_pipeline()

FileNotFoundError: Input TSV dataset not found at: /kaggle/input/datasets/jarroud/fake-news/multimodal_train.tsv